# parameter-subclass-of-tensor — ex1: Parameter subclasses MiniTensor with requires_grad=True default

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `parameter-subclass-of-tensor`. Running the final beacon cell reports progress against the `Backprop: Parameter subclasses Tensor` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Parameter subclasses Tensor` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`parameter-subclass-of-tensor`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "parameter-subclass-of-tensor"
DD_SUBTOPIC = "Backprop: Parameter subclasses Tensor"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Parameter subclasses Tensor — quick refresher

`nn.Parameter` is a `Tensor` subclass whose only difference is the default value of `requires_grad` — it's `True`, not `False`:

```python
class Parameter(Tensor):
    def __init__(self, array, requires_grad: bool = True):
        super().__init__(array, requires_grad=requires_grad)
```

Two rules:
- **Subclass, don't compose.** A `Parameter` IS-A `Tensor`. Every op   that takes `Tensor` accepts a `Parameter` automatically — no   conversion needed.
- **`isinstance(p, Tensor)` returns True for Parameters.** Critical   because `build_parents` and `unbox_args` both `isinstance(a,   Tensor)` — Parameters would be silently dropped if they weren't   Tensor subclasses.

The signaling value of the class itself: `isinstance(x, Parameter)` is how `nn.Module.parameters()` distinguishes the trainable state from the (Tensor-typed) buffers / activations. Same array data — different role, encoded purely through the class.

### Exercise 1 — Parameter subclasses MiniTensor with requires_grad=True default

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply the Parameter IS-A Tensor pattern: subclass MiniTensor with requires_grad=True as the default, preserving isinstance compatibility so optimizer/get_children find it.
> Keywords: parameter, subclass, requires-grad, default, is-a
> ```

**KCs targeted:** `parameter-subclass-of-tensor`, `get-children-callable-param`

Define `class Parameter(MiniTensor)` — a tiny subclass whose ONLY behavioral difference is the default value of `requires_grad`:

```python
class Parameter(MiniTensor):
    def __init__(self, array, requires_grad: bool = True):
        super().__init__(array, requires_grad=requires_grad)
```

Rules:

**1. Subclass, don't compose.** `Parameter` IS-A `MiniTensor`. Every op accepting a `MiniTensor` accepts a `Parameter` automatically — `multiply(p, t)` works without any conversion.

**2. `requires_grad=True` is the default.** Trainable params ALWAYS need grad. If the user omits the kwarg, the default True kicks in.

**3. Allow override.** `Parameter(arr, requires_grad=False)` must work — used for frozen layers / fine-tuning.

**4. `isinstance(p, MiniTensor)` is True for any `Parameter`.** This is the LOAD-BEARING property: `build_parents`, `unbox_args`, `get_children` all use `isinstance(_, MiniTensor)` as their gate. If `Parameter` were a separate class (composition not inheritance), every trainable param would be silently skipped by those helpers — the whole autograd layer would ignore your model's parameters.

**5. The class itself signals role.** `isinstance(x, Parameter)` is how a future `parameters()` walker would distinguish 'trainable state' from 'intermediate Tensor'. Same data, different role — encoded purely through the class.

In [ ]:
class Parameter(MiniTensor):
    """Subclass MiniTensor with requires_grad=True as the default."""
    def __init__(self, array, requires_grad: bool = True):
        raise NotImplementedError()



def _test_ex1():
    # --- Parameter is a subclass of MiniTensor ---
    assert issubclass(Parameter, MiniTensor), (
        'Parameter must subclass MiniTensor, not compose with it'
    )

    # --- default requires_grad=True ---
    p = Parameter(t.zeros(3))
    assert p.requires_grad is True, (
        'Parameter default must be requires_grad=True, '
        f'got {p.requires_grad}'
    )
    assert isinstance(p, MiniTensor), 'Parameter instance must be a MiniTensor too'
    assert isinstance(p, Parameter)

    # --- override to False (frozen layer) ---
    p_frozen = Parameter(t.zeros(3), requires_grad=False)
    assert p_frozen.requires_grad is False, (
        f'override requires_grad=False must work, got {p_frozen.requires_grad}'
    )

    # --- .array stored as-is ---
    raw = t.tensor([1.0, 2.0, 3.0])
    p2 = Parameter(raw)
    assert p2.array is raw, 'Parameter must store the raw tensor (identity, not copy)'

    # --- .recipe is None at construction (leaves carry no Recipe) ---
    assert p2.recipe is None, 'fresh Parameter is a leaf — no Recipe'

    # --- .grad starts as None (will be set by accumulate_grad later) ---
    assert p2.grad is None, 'fresh Parameter grad must start as None'

    # --- Parameter passes the isinstance(_, MiniTensor) gate used by helpers ---
    # This is the load-bearing test: every wrapper helper filters by isinstance(_, MiniTensor),
    # so Parameters must pass that test or they get silently skipped.
    def _build_parents(args):
        return {idx: a for idx, a in enumerate(args) if isinstance(a, MiniTensor)}

    x = MiniTensor(t.zeros(3))
    parents = _build_parents((x, p))
    assert parents == {0: x, 1: p}, (
        'Parameter must be picked up by isinstance(_, MiniTensor) — '
        f'got {parents}'
    )

    # --- the role signal: isinstance(x, Parameter) distinguishes from a plain MiniTensor ---
    assert isinstance(p, Parameter)
    assert not isinstance(x, Parameter), (
        'plain MiniTensor must NOT pass isinstance(_, Parameter) — '
        'subclassing must not pollute the parent class'
    )
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
class Parameter(MiniTensor):
    def __init__(self, array, requires_grad: bool = True):
        super().__init__(array, requires_grad=requires_grad)
```

**Why a subclass with only a default change.** It buys two things at near-zero cost:
1. **Default value.** `Parameter(t.zeros(3))` is the common case (trainable weight); kwargs default of True saves the user from writing `requires_grad=True` on every line of `__init__`.
2. **Type-as-tag.** `isinstance(x, Parameter)` is the only reliable way for `parameters()` to distinguish trainable state from incidental tensors. Same `.array`, same `.recipe`, same `.requires_grad` semantics — just a typing marker.

**Why NOT composition.** A composition design (`class Parameter: def __init__(self, t): self.tensor = t`) would mean `isinstance(p, MiniTensor)` is False. Every helper in the wrapper layer (`build_parents`, `unbox_args`, `get_children`) filters by `isinstance(_, MiniTensor)` — and would silently skip every parameter. The autograd layer would ignore your model's weights — silent, devastating bug.

**PyTorch's actual design.** `torch.nn.Parameter(torch.Tensor)` — exactly this pattern. The class body in PyTorch is similarly minimal: just an override of `__new__` to handle the `requires_grad=True` default and a `__deepcopy__` for state-dict-friendly copying. The IS-A relationship is the load-bearing design choice.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()